# Convection: upwind, central and the CFL condition

Linear advection: $\partial u/\partial t + c\,\partial u/\partial x = 0$, on a periodic domain. A square wave should move without changing shape.

The Courant number is $C = c\,\Delta t/\Delta x$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
c, nx = 1.0, 200
x = np.linspace(0, 2, nx, endpoint=False)
dx = x[1] - x[0]
u0 = np.where((x > 0.25) & (x < 0.75), 1.0, 0.0)

def run(step, C, t_end=1.0):
    dt = C * dx / c
    u = u0.copy()
    for _ in range(int(round(t_end / dt))):
        u = step(u, C)
    return u

exact = np.roll(u0, int(round(1.0 / dx)))   # shifted by c * t_end = 1

**Exercise 1.** Write the first-order upwind step for $c > 0$: $u_i^{n+1} = u_i^n - C\,(u_i^n - u_{i-1}^n)$. Use `np.roll` for the periodic neighbour.

In [ ]:
def upwind(u, C):
    ### BEGIN SOLUTION
    return u - C * (u - np.roll(u, 1))
    ### END SOLUTION

**Exercise 2.** Write the central (FTCS) step: $u_i^{n+1} = u_i^n - \tfrac{C}{2}\,(u_{i+1}^n - u_{i-1}^n)$.

In [ ]:
def central(u, C):
    ### BEGIN SOLUTION
    return u - 0.5 * C * (np.roll(u, -1) - np.roll(u, 1))
    ### END SOLUTION

**Exercise 3.** Lax–Wendroff is second order and stable for $C \le 1$:
$u_i^{n+1} = u_i^n - \tfrac{C}{2}(u_{i+1}-u_{i-1}) + \tfrac{C^2}{2}(u_{i+1} - 2u_i + u_{i-1})$.

In [ ]:
def lax_wendroff(u, C):
    # YOUR CODE HERE
    raise NotImplementedError

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, (name, step) in zip(axes, [("upwind", upwind), ("central", central), ("Lax-Wendroff", lax_wendroff)]):
    ax.plot(x, exact, "k", lw=3, alpha=0.3, label="exact")
    ax.plot(x, run(step, C=0.5), label="C = 0.5")
    ax.set_title(name); ax.set_ylim(-0.5, 1.5); ax.set_xlabel("x")
axes[0].legend();

What you should see:

* **Upwind** smears the edges. This is numerical diffusion.
* **Central** oscillates and blows up. FTCS is unconditionally unstable for pure advection.
* **Lax–Wendroff** keeps the edges sharper but wiggles next to them.

**Exercise 4.** Run upwind with $C = 1.0$ and $C = 1.1$. Explain both results in your notes.

In [ ]:
### BEGIN SOLUTION
for C in (1.0, 1.1):
    err = np.abs(run(upwind, C) - exact).max()
    print(f"upwind, C = {C}: max error = {err:.3g}")
### END SOLUTION